# 🇵🇰 Khaadi Content Detective
**Business question:** *What type of social-media content gets Pakistani audiences to engage and respond positively to Khaadi?*

**Flow:** Load → Inspect → Clean → Sentiment → Validate → EDA → Insights → AI-assisted recommendation

> ⚠️ **Practice mode:** the default file `khaadi_SAMPLE.csv` contains **made-up** comments. Use it only to test this notebook.
> When your real data is ready, change `DATA_FILE` in the next cell, run everything again, and write your insights **only from the real results**.

## 0. Settings

In [ ]:
DATA_FILE = "khaadi_labelled.csv"      # real data (sentiment labelled). Use "khaadi_SAMPLE.csv" for the fake practice file
USE_TRANSFORMER = True               # True = multilingual AI model (needs internet, e.g. in Colab)
                                     # False (or if the model can't load) = simple keyword method
MANUAL_LABEL_FILE = "manual_label_sheet.csv"   # created in section 5, you fill it by hand

## 1. Import libraries

In [ ]:
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 80)

## 2. Load and inspect the raw data

In [ ]:
raw = pd.read_csv(DATA_FILE, encoding="utf-8-sig")
print("Shape (rows, columns):", raw.shape)
raw.head(10)

In [ ]:
raw.info()

In [ ]:
print("Missing values per column:")
print(raw.isna().sum())
print("\nExact duplicate rows:", raw.duplicated().sum())
print("\nUnique platform values (look for spelling/case problems):")
print(raw["platform"].value_counts(dropna=False))

## 3. Clean the data

In [ ]:
df = raw.copy()

# 3.1 Standardise text columns (trim spaces, consistent case)
for c in ["platform", "post_category", "comment_text"]:
    df[c] = df[c].str.strip()
df["platform"] = df["platform"].str.title().replace({"Youtube": "YouTube"})
df["post_category"] = df["post_category"].str.lower()

# 3.2 Remove empty comments and exact duplicates
before = len(df)
df["comment_text"] = df["comment_text"].replace("", np.nan)
df = df.dropna(subset=["comment_text"])
df = df.drop_duplicates().reset_index(drop=True)
print(f"Rows: {before} -> {len(df)} (removed {before - len(df)})")

# 3.3 Fix data types
if "post_date" in df.columns:
    df["post_date"] = pd.to_datetime(df["post_date"], errors="coerce")
df["likes_on_comment"] = pd.to_numeric(df["likes_on_comment"], errors="coerce")
# Blank likes usually means 'not visible', so we assume 0 (a choice we mention in the report)
df["likes_on_comment"] = df["likes_on_comment"].fillna(0).clip(lower=0).astype(int)

print(df.dtypes)
print("\nPlatforms now:", df["platform"].unique())

### 3.4 Derived features

In [ ]:
URDU_SCRIPT = re.compile(r"[\u0600-\u06FF]")
ROMAN_URDU_WORDS = {"bohat","hai","hain","nahi","nahin","kya","mein","acha","acchi","kharab","mehngi","mehnga",
                    "karein","aayi","zabardast","khoobsurat","kamal","mashallah","pehle","jaisi","ye","aur","kar","ho"}

def detect_language(t):
    if URDU_SCRIPT.search(t):
        return "urdu"
    words = set(re.findall(r"[a-z]+", t.lower()))
    return "roman_urdu" if words & ROMAN_URDU_WORDS else "english"

if "language" not in df.columns:
    df["language"] = df["comment_text"].apply(detect_language)       # heuristic, check by hand
df["clean_text"] = df["comment_text"].str.replace(r"\[SAMPLE\]", "", regex=True).str.strip()
df["text_length"] = df["clean_text"].str.len()
if "post_date" in df.columns:
    df["day_of_week"] = df["post_date"].dt.day_name()
    df["month"] = df["post_date"].dt.to_period("M").astype(str)
HAS_DATES = "post_date" in df.columns and df["post_date"].notna().sum() >= 10
df["is_sample"] = df["comment_text"].str.contains(r"\[SAMPLE\]")

if df["is_sample"].any():
    print("⚠️  This is SAMPLE (fake) data. Do not use these results as real findings.")
df.head()

## 4. Sentiment analysis
Pakistani comments mix English, Urdu script and Roman Urdu (*"bohat mehngi hai"*), so ordinary tools like VADER often call Roman Urdu neutral.
Two methods are provided:
1. **Multilingual transformer** (`cardiffnlp/twitter-xlm-roberta-base-sentiment`), best choice in Colab.
2. **Keyword method**, a small English / Roman Urdu / Urdu word list. Simple and runs anywhere, but it is crude.

In [ ]:
POS = ["love","beautiful","best","gorgeous","superb","perfect","great","lovely","helpful","fast","worth","stunning",
       "amazing","favourite","favorite","acha","acchi","khoobsurat","zabardast","kamal","mashallah","behtareen",
       "خوبصورت","بہترین"]
NEG = ["mehng","kharab","late","disappoint","wrong","terrible","rude","poor","overpriced","fade","cancel","worst",
       "bad","zyada","bakwas","ghatia","nahi","nahin","expensive","return","مہنگ","نہیں"]

def keyword_sentiment(text):
    t = text.lower()
    p = sum(bool(re.search(r"\b" + w, t)) for w in POS)
    n = sum(bool(re.search(r"\b" + w, t)) for w in NEG)
    return "positive" if p > n else "negative" if n > p else "neutral"

def transformer_sentiment(texts):
    from transformers import pipeline
    clf = pipeline("sentiment-analysis", model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
                   truncation=True, max_length=256)
    return [r["label"].lower() for r in clf(list(texts), batch_size=32)]

method = "keyword"
if "sentiment" in df.columns and df["sentiment"].astype(str).str.strip().ne("").all():
    df["sentiment"] = df["sentiment"].str.lower().str.strip()
    method = "pre-labelled (Claude-assisted labels, validate by hand)"
elif USE_TRANSFORMER:
    try:
        df["sentiment"] = transformer_sentiment(df["clean_text"])
        method = "transformer"
    except Exception as e:
        print("Transformer not available, using keyword method instead.\nReason:", str(e)[:120])
if method == "keyword" and "sentiment" not in df.columns:
    df["sentiment"] = df["clean_text"].apply(keyword_sentiment)

print("Method used:", method)
df["sentiment"].value_counts()

## 5. Validate by hand (this makes the project stand out)
Run the next cell to export 50 random comments. Open `manual_label_sheet.csv`, fill the `manual_label` column with
`positive`, `negative` or `neutral`, save it, then re-run the second cell to see how accurate the model is.

In [ ]:
sample = df.sample(min(50, len(df)), random_state=1)[["comment_text", "language", "sentiment"]].copy()
sample["manual_label"] = ""
sample.to_csv(MANUAL_LABEL_FILE, index=False, encoding="utf-8-sig")
print("Saved", MANUAL_LABEL_FILE, "- fill the manual_label column, then run the next cell.")

In [ ]:
import os
if os.path.exists(MANUAL_LABEL_FILE):
    m = pd.read_csv(MANUAL_LABEL_FILE, encoding="utf-8-sig")
    m = m[m["manual_label"].isin(["positive", "negative", "neutral"])]
    if len(m) == 0:
        print("manual_label column is still empty. Fill it by hand first.")
    else:
        m["correct"] = m["manual_label"] == m["sentiment"]
        print(f"Labelled rows: {len(m)} | Accuracy: {m['correct'].mean():.0%}")
        print("\nAccuracy by language (look for weakness on Roman Urdu):")
        print(m.groupby("language")["correct"].agg(["mean", "count"]).round(2))
        print("\nConfusion table (rows = your label, columns = model):")
        print(pd.crosstab(m["manual_label"], m["sentiment"]))

## 6. Exploratory data analysis

### 6.1 Overall sentiment

In [ ]:
order = ["positive", "neutral", "negative"]
colors = {"positive": "#2a9d8f", "neutral": "#b0b0b0", "negative": "#e76f51"}
counts = df["sentiment"].value_counts().reindex(order).fillna(0)
plt.figure(figsize=(5, 5))
plt.pie(counts, labels=counts.index, autopct="%1.1f%%", colors=[colors[k] for k in counts.index], startangle=90)
plt.title("Overall sentiment")
plt.show()

### 6.2 Comment volume by platform

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="platform", order=df["platform"].value_counts().index, color="#457b9d")
plt.title("Comments collected by platform")
plt.show()

### 6.3 Sentiment by platform

In [ ]:
ct = pd.crosstab(df["platform"], df["sentiment"], normalize="index").reindex(columns=order).fillna(0) * 100
ct.plot(kind="bar", stacked=True, color=[colors[c] for c in ct.columns], figsize=(7, 4))
plt.ylabel("% of comments"); plt.title("Sentiment mix by platform"); plt.xticks(rotation=0)
plt.legend(title="")
plt.show()
ct.round(1)

### 6.4 Sentiment by post category (answers 'which content type works?')

In [ ]:
cat = pd.crosstab(df["post_category"], df["sentiment"], normalize="index").reindex(columns=order).fillna(0) * 100
cat = cat.sort_values("positive", ascending=False)
cat.plot(kind="barh", stacked=True, color=[colors[c] for c in cat.columns], figsize=(8, 4))
plt.xlabel("% of comments"); plt.title("Sentiment mix by post category"); plt.legend(title="")
plt.show()
cat.round(1)

### 6.5 Do liked comments lean positive or negative?

In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(data=df, x="sentiment", y="likes_on_comment", order=order, palette=colors, errorbar=None)
plt.title("Average likes per comment by sentiment")
plt.show()
df.groupby("sentiment")["likes_on_comment"].agg(["mean", "median", "count"]).round(2)

### 6.6 Language mix

In [ ]:
lang = pd.crosstab(df["language"], df["sentiment"]).reindex(columns=order).fillna(0)
lang.plot(kind="bar", stacked=True, color=[colors[c] for c in lang.columns], figsize=(6, 4))
plt.title("Language of comments and sentiment"); plt.xticks(rotation=0)
plt.show()

### 6.7 Themes customers talk about (price, quality, delivery, sizing, service)

In [ ]:
THEMES = {
    "price":    ["mehng", "price", "rupee", "overpriced", "expensive", "مہنگ"],
    "quality":  ["quality", "fabric", "stitch", "fade", "کوالٹی", "کپڑے"],
    "delivery": ["delivery", "late", "packaging", "order", "cancel"],
    "sizing":   ["size", "sizing", "fit"],
    "service":  ["staff", "service", "rude", "helpful", "reply"],
}
rows = []
for theme, words in THEMES.items():
    mask = df["clean_text"].str.lower().apply(lambda t: any(w in t for w in words))
    for s in order:
        rows.append({"theme": theme, "sentiment": s, "comments": int((mask & (df["sentiment"] == s)).sum())})
theme_df = pd.DataFrame(rows).pivot(index="theme", columns="sentiment", values="comments")[order]
theme_df.plot(kind="bar", color=[colors[c] for c in theme_df.columns], figsize=(8, 4))
plt.title("Themes mentioned, split by sentiment"); plt.xticks(rotation=0)
plt.show()
theme_df

### 6.8 Most common words in positive vs negative comments

In [ ]:
STOP = {"the","is","a","of","to","in","and","this","it","i","for","my","was","are","on","will","be","at","me","you",
        "hai","hain","ka","ki","ke","ye","se","ko","kya","mein","bhi","pe","aur","please","sample"}
def top_words(s, n=10):
    words = re.findall(r"[a-zA-Z\u0600-\u06FF]{3,}", " ".join(df.loc[df["sentiment"] == s, "clean_text"]).lower())
    return Counter(w for w in words if w not in STOP).most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, s in zip(axes, ["positive", "negative"]):
    tw = top_words(s)
    if tw:
        w, c = zip(*tw)
        ax.barh(w[::-1], c[::-1], color=colors[s])
    ax.set_title(f"Top words: {s}")
plt.tight_layout(); plt.show()

### 6.9 Timing: day of week and month

In [ ]:
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
if not HAS_DATES:
    print("This dataset has no post dates, so the timing charts are skipped.")
else:
    dow = df.groupby("day_of_week").agg(comments=("comment_text", "count"),
                                        avg_likes=("likes_on_comment", "mean"),
                                        pct_positive=("sentiment", lambda s: (s == "positive").mean() * 100)).reindex(days).dropna()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    dow["avg_likes"].plot(kind="bar", ax=axes[0], color="#457b9d"); axes[0].set_title("Average likes by post day")
    dow["pct_positive"].plot(kind="bar", ax=axes[1], color="#2a9d8f"); axes[1].set_title("% positive by post day")
    plt.tight_layout(); plt.show()

In [ ]:
if not HAS_DATES:
    print("No post dates, skipped.")
else:
    monthly = df.dropna(subset=["post_date"]).groupby("month")["likes_on_comment"].mean()
    plt.figure(figsize=(8, 3.5)); monthly.plot(marker="o", color="#457b9d")
    plt.title("Average comment likes by month"); plt.xticks(rotation=45)
    plt.show()

## 7. Summary tables for the final report

In [ ]:
summary = (df.groupby("post_category")
             .agg(comments=("comment_text", "count"),
                  pct_positive=("sentiment", lambda s: round((s == "positive").mean() * 100, 1)),
                  pct_negative=("sentiment", lambda s: round((s == "negative").mean() * 100, 1)),
                  avg_likes=("likes_on_comment", lambda s: round(s.mean(), 2)))
             .sort_values("pct_positive", ascending=False))
summary

In [ ]:
platform_summary = (df.groupby("platform")
                      .agg(comments=("comment_text", "count"),
                           pct_positive=("sentiment", lambda s: round((s == "positive").mean() * 100, 1)),
                           pct_negative=("sentiment", lambda s: round((s == "negative").mean() * 100, 1)),
                           avg_likes=("likes_on_comment", lambda s: round(s.mean(), 2)))
                      .sort_values("pct_positive", ascending=False))
platform_summary

## 7b. Answers to the project questions
These tables answer the questions from the project story. Engagement here = likes on a comment (post-level reach is not visible).

In [ ]:
def table(g):
    return g.agg(comments=("comment_text", "count"),
                 pct_pos=("sentiment", lambda s: round((s == "positive").mean() * 100, 1)),
                 pct_neg=("sentiment", lambda s: round((s == "negative").mean() * 100, 1)),
                 median_likes=("likes_on_comment", "median"),
                 pct_liked=("likes_on_comment", lambda s: round((s >= 1).mean() * 100, 1)))

print("Q1. Which platform?");            display(table(df.groupby("platform")))
print("Q2. Which content type?");        display(table(df.groupby("post_category")))
print("Q4. Which sentiment gets engagement?")
display(df.groupby("sentiment")["likes_on_comment"].agg(["count", "median", "sum"]))
top = df["likes_on_comment"].idxmax()
print("Total likes by sentiment WITHOUT the single most-liked comment:")
display(df.drop(index=top).groupby("sentiment")["likes_on_comment"].sum())
print("Q6. Which audience segment (language)?"); display(table(df.groupby("language")))

In [ ]:
if "theme" in df.columns:
    print("Q3. Which topics?")
    display(table(df.groupby("theme")).sort_values("comments", ascending=False))
print("Hashtags used:", df["comment_text"].str.findall(r"#\w+").explode().dropna().tolist())
print("Overseas / shipping mentions:", df["comment_text"].str.contains(r"\bUK\b|overseas|worldwide|USA", case=False).sum())
print("Q5. Time/day:", "available" if HAS_DATES else "not available (this dataset has no post dates)")

## 8. AI-assisted marketing recommendation
The next cell builds a prompt from your **summary tables** (not the raw comments). Copy the printed text into Claude or ChatGPT,
then **check every claim in the answer against your own tables** before putting it in your report.

In [ ]:
prompt = f"""You are a digital-marketing analyst for Khaadi, a Pakistani fashion brand.
Below are sentiment results from {len(df)} public comments/reviews (method: {method}).

Results by post category:
{summary.to_string()}

Results by platform:
{platform_summary.to_string()}

Task: suggest THREE post ideas for Khaadi's next campaign. For each idea give the platform, post category,
and a justification that cites specific numbers from the tables. Also mention one weakness in the data
(for example small sample size or Roman Urdu errors) that limits how confident we can be."""
print(prompt)

## 9. Insights (write these from your REAL data results)
*Delete this note and the sample wording once you switch to real data.*

1. **Best-performing content type:** …
2. **Platform that responds most positively:** …
3. **Main complaints (themes):** …
4. **Does negativity get more likes?** …
5. **Best posting day (if any pattern):** …
6. **Limitations:** sample size, manual collection, Roman Urdu accuracy of ___%.

## 10. Conclusion & recommendation
…

## 11. Ethics note
Comments were collected manually from public posts, usernames were not stored, and the data is used only for a class project.

## 12. Chatbot interface (run this LAST)
When every cell above has finished (sentiment included), run the next two cells. A **public link** appears (`https://....gradio.live`) that you can open on your phone or share with your teacher.
The page has the 7 project questions on the left. Tap one, or type a question, and the answers are **calculated live from your data**, so they always match your tables.
The link works while this Colab session is running (up to 1 week). Re-run the cell to get a new one.

In [ ]:
%pip install -q "gradio>=5"

In [ ]:
import gradio as gr

SHARE = True          # True = creates a public link you can open anywhere
MIN_PLATFORM = 30     # platforms with fewer comments are flagged as too few to judge
MIN_CAT = 10          # same for content categories
MIN_SEG = 10          # same for audience segments

# ---- make sure every row has a theme (reuses the 'theme' column if it exists) ----
def theme_of(t, sent):
    x = t.lower()
    if re.search(r"sleeve|slvles|modest|chaddy", x): return "sleeveless_modesty"
    if re.search(r"price|expensive|kidney|cost|afford|paisy|mehng|mhnga|kimat|reasonable|how much|\bpp\b|paisa|worth|\brs\b|jaidad", x): return "price"
    if re.search(r"link|code|website|order|where|available|shipping|worldwide|\bcod\b|sold out|sale|find|stock|item number|\buk\b|overseas|location|kab", x): return "availability_ordering"
    if re.search(r"quality|fabric|stitch|defective|torn|fit|pyjama|neck|damage|demage|print|colou?r|cuts|design|lawn|cambric|material|bedsheet", x): return "product_design_quality"
    if re.search(r"customer service|service|manager|parcel", x): return "customer_service"
    return "praise_aesthetic" if sent == "positive" else "other_opinion"

if "theme" not in df.columns:
    df["theme"] = [theme_of(t, s) for t, s in zip(df["comment_text"], df["sentiment"])]

# ---- helpers ----
def _fmt(v):
    return f"{v:g}" if isinstance(v, float) else str(v)

def md_table(t):
    t = t.reset_index()
    head = "| " + " | ".join(map(str, t.columns)) + " |\n|" + "---|" * len(t.columns) + "\n"
    rows = "".join("| " + " | ".join(_fmt(v) for v in r) + " |\n" for r in t.itertuples(index=False))
    return head + rows

def _tbl(g):
    return g.agg(**{"comments": ("comment_text", "count"),
                    "% positive": ("sentiment", lambda s: round((s == "positive").mean() * 100, 1)),
                    "% negative": ("sentiment", lambda s: round((s == "negative").mean() * 100, 1)),
                    "median likes": ("likes_on_comment", "median"),
                    "% liked": ("likes_on_comment", lambda s: round((s >= 1).mean() * 100, 1))})

def _pretty(t, name=""):
    t = t.copy(); t.index = [str(i).replace("_", " ") for i in t.index]; t.index.name = name; return t

CAVEAT = "\n\n_Engagement = likes on a comment (post reach is not visible). Sentiment labels should be validated by hand._"

# ---- the 7 answers (all calculated from df) ----
def a_overview():
    n = len(df); vc = df["sentiment"].value_counts(normalize=True).mul(100).round(1)
    return (f"**{n} comments analysed** ({', '.join(f'{k}: {v}' for k, v in df['platform'].value_counts().items())}).\n\n"
            f"Overall sentiment: **{vc.get('positive', 0)}% positive**, {vc.get('neutral', 0)}% neutral, **{vc.get('negative', 0)}% negative**.\n\n"
            "Tap a question on the left, or type one.")

def a_platform():
    t = _tbl(df.groupby("platform")).sort_values("comments", ascending=False)
    out = "### Which platform performs better?\n\n" + md_table(t) + "\n"
    ok, few = t[t["comments"] >= MIN_PLATFORM], t[t["comments"] < MIN_PLATFORM]
    if len(ok) == 1:
        out += f"**Answer:** only **{ok.index[0]}** has enough comments ({int(ok['comments'].iloc[0])}) for a fair judgement."
    elif len(ok) >= 2:
        e, p = ok["% liked"].idxmax(), ok["% positive"].idxmax()
        out += (f"**Answer:** **{e}** gets the most engagement ({ok.loc[e, '% liked']}% of comments liked); "
                f"**{p}** is the most positive ({ok.loc[p, '% positive']}% positive).")
    else:
        out += "**Answer:** no platform has enough comments yet for a fair comparison."
    if len(few):
        out += f"\n\n⚠️ {', '.join(few.index)} have fewer than {MIN_PLATFORM} comments, so they are not compared."
    return out + CAVEAT

def a_content():
    main = df["platform"].value_counts().idxmax()
    t = _tbl(df[df["platform"] == main].groupby("post_category")).sort_values("comments", ascending=False)
    out = f"### Which content type gets more engagement?\n\nCompared within **{main}** (the platform with the most comments).\n\n" + md_table(_pretty(t, "post category")) + "\n"
    ok = t[t["comments"] >= MIN_CAT]
    if len(ok) >= 2:
        top = ok.sort_values(["median likes", "% liked"], ascending=False).index[0]
        liked, warm, crit = ok["% liked"].idxmax(), ok["% positive"].idxmax(), ok["% negative"].idxmax()
        s1 = (f"**{top.replace('_', ' ')}** gets the most engagement (median {_fmt(float(ok.loc[top, 'median likes']))} likes per comment, "
              f"{ok.loc[top, '% liked']}% of comments liked). ")
        if liked != top:
            s1 += f"**{liked.replace('_', ' ')}** has the most comments that got a like ({ok.loc[liked, '% liked']}%). "
        out += ("**Answer:** " + s1 +
                f"**{warm.replace('_', ' ')}** is the warmest ({ok.loc[warm, '% positive']}% positive) and "
                f"**{crit.replace('_', ' ')}** the most critical ({ok.loc[crit, '% negative']}% negative).")
    else:
        out += "**Answer:** not enough categories with 10 or more comments yet."
    small = t[t["comments"] < MIN_CAT]
    if len(small):
        out += f"\n\n⚠️ Too few comments to judge: {', '.join(i.replace('_', ' ') for i in small.index)}."
    return out + CAVEAT

def a_topics():
    t = _tbl(df.groupby("theme")).sort_values("comments", ascending=False)
    tags = df["comment_text"].str.findall(r"#\w+").explode().dropna().str.lower()
    out = "### What topics and hashtags are popular?\n\n" + md_table(_pretty(t, "topic")) + "\n"
    out += (f"**Hashtags:** {len(tags)} found in {len(df)} comments" + (" (" + ", ".join(f"{k} x{v}" for k, v in tags.value_counts().head(5).items()) + ")" if len(tags) else "") +
            ", so topics are more useful than tags.\n\n")
    issues = t.drop(index=[i for i in ["praise_aesthetic", "other_opinion"] if i in t.index])
    big = t.loc[t["comments"] >= 5, "median likes"]
    out += (f"**Answer:** the biggest topic is **{t.index[0].replace('_', ' ')}** ({int(t['comments'].iloc[0])} comments). "
            f"The biggest concern is **{issues['comments'].idxmax().replace('_', ' ')}** ({int(issues['comments'].max())} comments, "
            f"{issues.loc[issues['comments'].idxmax(), '% negative']}% negative). "
            f"The most-liked topic is **{big.idxmax().replace('_', ' ')}** (median {_fmt(float(big.max()))} likes).")
    return out + CAVEAT

def a_sentiment():
    g = df.groupby("sentiment")["likes_on_comment"]
    t = pd.DataFrame({"comments": g.count(), "median likes": g.median(),
                      "% liked": g.apply(lambda s: round((s >= 1).mean() * 100, 1)),
                      "total likes": g.sum()}).reindex(["positive", "neutral", "negative"]).dropna()
    top = df["likes_on_comment"].idxmax()
    wo = df.drop(index=top).groupby("sentiment")["likes_on_comment"].sum()
    best = t["median likes"].idxmax()
    out = "### What sentiment generates engagement?\n\n" + md_table(t) + "\n"
    out += (f"**Answer:** **{best}** comments get the most likes per comment (median {_fmt(float(t.loc[best, 'median likes']))}; "
            f"{t.loc[best, '% liked']}% got at least one like).\n\n"
            f"The single most-liked comment has {int(df.loc[top, 'likes_on_comment'])} likes ({df.loc[top, 'sentiment']}). "
            f"Without it, total likes are: " + ", ".join(f"{k} {int(v)}" for k, v in wo.reindex(['positive', 'neutral', 'negative']).dropna().items()) + ".")
    return out + CAVEAT

def a_time():
    out = "### What time or day appears better?\n\n"
    if not HAS_DATES or "day_of_week" not in df.columns:
        return out + ("**Answer:** this dataset has no post dates, so the best time or day cannot be measured.\n\n"
                      "Comment ages such as '4w' show when someone commented, not when the post went live, so they cannot be used. "
                      "This is a limitation of the study: add a post_date column to answer this question in future.")
    days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    d = df.dropna(subset=["post_date"])
    t = d.groupby("day_of_week").agg(**{"comments": ("comment_text", "count"),
                                       "median likes": ("likes_on_comment", "median"),
                                       "avg likes": ("likes_on_comment", lambda s: round(s.mean(), 1)),
                                       "% positive": ("sentiment", lambda s: round((s == "positive").mean() * 100, 1))}).reindex(days).dropna()
    ok = t[t["comments"] >= 5]
    out += md_table(t) + "\n"
    if len(ok):
        e, p = ok["median likes"].idxmax(), ok["% positive"].idxmax()
        out += f"**Answer:** **{e}** posts get the most likes per comment, and **{p}** posts are the most positive. Days with fewer than 5 comments are ignored."
    return out + CAVEAT

def a_audience():
    t = _tbl(df.groupby("language")).sort_values("comments", ascending=False)
    ok = t[t["comments"] >= MIN_SEG]
    out = "### Which Pakistani audience segment seems more responsive?\n\n" + md_table(_pretty(t, "language")) + "\n"
    if len(ok) >= 2:
        crit, warm = ok["% negative"].idxmax(), ok["% positive"].idxmax()
        out += (f"**Answer:** **{crit.replace('_', ' ')}** commenters are the most critical ({ok.loc[crit, '% negative']}% negative); "
                f"**{warm.replace('_', ' ')}** commenters are the most positive ({ok.loc[warm, '% positive']}%).\n\n")
    seg = df.groupby("theme").agg(n=("comment_text", "count"), neg=("sentiment", lambda s: int((s == "negative").sum())))
    ov = int(df["comment_text"].str.contains(r"\bUK\b|overseas|worldwide|USA", case=False).sum())
    for th, label in [("price", "Price-sensitive shoppers"), ("sleeveless_modesty", "Modest-fashion buyers"), ("availability_ordering", "Buyers who cannot find or order items")]:
        if th in seg.index:
            out += f"- **{label}:** {int(seg.loc[th, 'n'])} comments, {int(seg.loc[th, 'neg'])} negative\n"
    out += f"- **Overseas buyers:** {ov} comments mention the UK, overseas or worldwide shipping\n"
    return out + "\n_Language is detected with a simple word list, so treat the split as approximate._" + CAVEAT

def a_next():
    ts = df.groupby("theme").agg(n=("comment_text", "count"),
                                 neg=("sentiment", lambda s: int((s == "negative").sum())),
                                 neu=("sentiment", lambda s: int((s == "neutral").sum())),
                                 likes=("likes_on_comment", "sum"))
    ideas = {"availability_ordering": ("Shoppable post", "Put item codes or links in captions or a pinned comment, add a clear sale calendar and a shipping notice (including overseas)."),
             "price": ("Full-outfit price post", "Show the full set price upfront (shirt, dupatta, trouser) with fabric and value details."),
             "sleeveless_modesty": ("Sleeves-included styling post", "Show sleeved options or how to style the look modestly."),
             "product_design_quality": ("Quality-proof post", "Show fabric, fit and stitching close-ups and address repeated product complaints."),
             "customer_service": ("Customer-care post", "Show how complaints, returns and orders are handled.")}
    rows = []
    for th, (title, how) in ideas.items():
        if th in ts.index:
            r = ts.loc[th]
            pain = int(r["neg"] + (r["neu"] if th == "availability_ordering" else 0))
            if pain > 0:
                rows.append((pain, th, title, how, r))
    rows.sort(key=lambda x: -x[0])
    out = "### What should Khaadi post next?\n\n"
    for i, (pain, th, title, how, r) in enumerate(rows[:3], 1):
        out += (f"**{i}. {title}**\n{how}\n_Evidence: {int(r['n'])} comments on this topic, {int(r['neg'])} negative, "
                f"{int(r['neu'])} neutral, {int(r['likes'])} likes in total._\n\n")
    main = df["platform"].value_counts().idxmax()
    t = _tbl(df[df["platform"] == main].groupby("post_category"))
    t = t[t["comments"] >= MIN_CAT]
    if len(t):
        w = t["% positive"].idxmax()
        out += f"**Keep doing:** **{w.replace('_', ' ')}** posts on {main}, the warmest content ({t.loc[w, '% positive']}% positive, {t.loc[w, '% negative']}% negative).\n\n"
    out += (f"**Limits:** {len(df)} comments, mostly {main}, skewed to visible comments; "
            f"{'post dates available' if HAS_DATES else 'no post dates'}; sentiment method: {method}. Check these ideas against your own tables.")
    return out

QUESTIONS = [("1. Which platform performs better?", a_platform),
             ("2. Which content type gets more engagement?", a_content),
             ("3. What topics and hashtags are popular?", a_topics),
             ("4. What sentiment generates engagement?", a_sentiment),
             ("5. What time or day appears better?", a_time),
             ("6. Which Pakistani audience segment seems more responsive?", a_audience),
             ("7. What should a Pakistani brand post next?", a_next)]
EXACT = {q: f for q, f in QUESTIONS}
KEYS = [(a_platform, ["platform", "instagram", "youtube", "facebook", "app store", "channel"]),
        (a_content, ["content", "category", "type", "collection", "sale", "influencer"]),
        (a_topics, ["topic", "hashtag", "theme", "popular", "complain", "talk"]),
        (a_sentiment, ["sentiment", "positive", "negative", "emotion", "feel", "engage", "like"]),
        (a_time, ["time", "day", "when", "date", "week", "month", "hour"]),
        (a_audience, ["audience", "segment", "language", "urdu", "who", "responsive", "overseas", "customer"]),
        (a_next, ["next", "recommend", "should", "idea", "suggest", "campaign"])]

def route(q):
    if q in EXACT: return EXACT[q]()
    low = q.lower()
    if any(w in low for w in ["hello", "hi ", "overview", "summary", "help"]) or low.strip() in ("hi", "hey"):
        return a_overview()
    scores = [(sum(k in low for k in kws), f) for f, kws in KEYS]
    best = max(scores, key=lambda x: x[0])
    if best[0] == 0:
        return "I can answer the 7 project questions. Tap one on the left, or ask about: platform, content type, topics, sentiment, time or day, audience, or what to post next."
    return best[1]()

# ---- the interface ----
def make_chatbot(value):
    try:
        return gr.Chatbot(value=value, type="messages", height=560)
    except TypeError:
        return gr.Chatbot(value=value, height=560)

def make_click(q):
    def f(history):
        return (history or []) + [{"role": "user", "content": q}, {"role": "assistant", "content": route(q)}]
    return f

def on_submit(text, history):
    if not text.strip(): return history, ""
    return (history or []) + [{"role": "user", "content": text}, {"role": "assistant", "content": route(text)}], ""

WELCOME = [{"role": "assistant", "content": "**Welcome to the Khaadi Content Detective** 🇵🇰\n\n" + a_overview()}]

with gr.Blocks(title="Khaadi Content Detective") as demo:
    gr.Markdown("# 🇵🇰 Khaadi Content Detective\nWhat type of social-media content gets Pakistani audiences to engage?")
    with gr.Row():
        with gr.Column(scale=1, min_width=280):
            gr.Markdown("### Questions")
            buttons = [(gr.Button(q), q) for q, _ in QUESTIONS]
            clear = gr.Button("Clear chat")
        with gr.Column(scale=3):
            bot = make_chatbot(WELCOME)
            box = gr.Textbox(placeholder="Type a question, e.g. which platform is best?", show_label=False)
    for b, q in buttons:
        b.click(make_click(q), inputs=bot, outputs=bot)
    box.submit(on_submit, [box, bot], [bot, box])
    clear.click(lambda: WELCOME, None, bot)

demo.launch(share=SHARE, debug=False)